# Chapter 6: Data Loading, Storage, File Formats

## Reading and Writing Data in Text Format

In [1]:
# Import packages
import pandas as pd
import csv
import sys
import json
import numpy as np
import requests
import sqlite3
import os
import sqlalchemy as sqla
from lxml import objectify

pd.options.display.max_rows = 10

Example 1: `ex1.csv`, a csv file with headers.

In [2]:
with open("../examples/ex1.csv") as file:
    print(file.read())

a,b,c,d,message
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


This file can be read with `pandas.read_csv()` as follows:

In [3]:
df = pd.read_csv("../examples/ex1.csv")
df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Example 2: `ex2.csv`, a csv file with no headers.

In [4]:
with open("../examples/ex2.csv") as file:
    print(file.read())

1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


Without column headers, this file can be read with `pandas.read_csv()` as follows:
- With automatically assigned column names.
- With manually provided column names.

In [5]:
# Auto column names
df = pd.read_csv("../examples/ex2.csv", header=None)
df

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [6]:
# Manual column names
df = pd.read_csv("../examples/ex2.csv", names=["a", "b", "c", "d", "text"])
df

,a,b,c,d,text
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


A column in the file can be used as row labels. The column can be selected via index, original column names, or by provided column names.

In [7]:
# Provided column names
pd.read_csv("../examples/ex1.csv", index_col="message")

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


In [8]:
# Provided column names
pd.read_csv("../examples/ex2.csv", header=None, index_col=4)

,0,1,2,3
4,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


In [9]:
# Provided column names
pd.read_csv("../examples/ex2.csv", names=["a", "b", "c", "d", "text"], index_col="text")

,a,b,c,d
text,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


Multiple columns can be used for row names, creating a hierarchical index.

In [10]:
# View example data
with open("../examples/csv_mindex.csv") as file:
    print(file.read())

key1,key2,value1,value2
one,a,1,2
one,b,3,4
one,c,5,6
one,d,7,8
two,a,9,10
two,b,11,12
two,c,13,14
two,d,15,16



In [11]:
# Import
pd.read_csv("../examples/csv_mindex.csv", index_col=["key1", "key2"])

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

Make sure to add the keys in the correct order.

In [12]:
# Import
pd.read_csv("../examples/csv_mindex.csv", index_col=["key2", "key1"])

,,value1,value2
key2,key1,,
a,one,1,2
b,one,3,4
c,one,5,6
d,one,7,8
a,two,9,10
b,two,11,12
c,two,13,14
d,two,15,16


In case the file doesn't have fixed delimiters, use regex. For example:

In [13]:
# View example data
with open("../examples/ex3.txt") as file:
    print(file.read())

            A         B         C
aaa -0.264438 -1.026059 -0.619500
bbb  0.927272  0.302904 -0.032399
ccc -0.264273 -0.386314 -0.217601
ddd -0.871858 -0.348382  1.100491



The header contains of tabs and whitespaces; and the number of whitespaces differs between the header and rest of the table. We can read this table by using variable whitespaces as delimiter.

In [14]:
pd.read_table("../examples/ex3.txt", sep="\\s+")

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


In case there are rows to be skipped, use the `skiprows` argument.

In [15]:
with open("../examples/ex4.csv") as file:
    print(file.read())

# hey!
a,b,c,d,message
# just wanted to make things more difficult for you
# who reads CSV files with computers, anyway?
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo



In [16]:
pd.read_csv("../examples/ex4.csv", skiprows=[0, 2, 3])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Handling of missing data in Python is very flexible:
- By default, missing values or sentinel values often used to represent missing values (NA, NULL) are treated as `NaN`.
- This can be ignored, importing all values as is.
- A custom set of sentinel values can be applied:
    - Across the entire table
    - Or on a column-by-column basis

In [17]:
# Example data
with open("../examples/ex5.csv") as file:
    print(file.read())

something,a,b,c,d,message
one,1,2,3,4,NA
two,5,6,,8,world
three,9,10,11,12,foo


In [18]:
# Default settings
pd.read_csv("../examples/ex5.csv")

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [19]:
# Import all values as is
pd.read_csv("../examples/ex5.csv", keep_default_na=False)

,something,a,b,c,d,message
0,one,1,2,3,4,NA
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [20]:
# Import values with custom NA rules across dataframe
pd.read_csv("../examples/ex5.csv", keep_default_na=False, na_values="NA")

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [21]:
# Import values with custom NA rules across columns using a dictionary
pd.read_csv(
    "../examples/ex5.csv",
    keep_default_na=False,
    na_values={"c": [""], "message": ["foo", "world"]},
)

,something,a,b,c,d,message
0,one,1,2,3.0,4,NA
1,two,5,6,NaN,8,NaN
2,three,9,10,11.0,12,NaN


### Reading Text Files in Pieces

Files can be read in chunks for easier processing, or only read in a small amount (the head) while processing arguments are not yet finalised.

In [22]:
# Example data
ex = pd.read_csv("../examples/ex6.csv")

In [23]:
# Read only starting rows
pd.read_csv("../examples/ex6.csv", nrows=10)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
5,1.817480,0.742273,0.419395,-2.251035,Q
6,-0.776764,0.935518,-0.332872,-1.875641,U
7,-0.913135,1.530624,-0.572657,0.477252,K
8,0.358480,-0.497572,-0.367016,0.507702,S
9,-1.740877,-1.160417,-1.637830,2.172201,G


In [24]:
# Read in chunks
chunks = pd.read_csv("../examples/ex6.csv", chunksize=1000)
chunks

The chunks object can be viewed chunk by chunk using the `get_chunk` method. Note that the method is imperative.

In [25]:
# Default chunk size as defined
chunks.get_chunk()

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
995,2.311896,-0.417070,-1.409599,-0.515821,M
996,-0.479893,-0.650419,0.745152,-0.646038,H
997,0.523331,0.787112,0.486066,1.093156,D
998,-0.362559,0.598894,-1.843201,0.887292,W


In [26]:
# Selected chunk size
chunks.get_chunk(100)

,one,two,three,four,key
1000,0.467976,-0.038649,-0.295344,-1.824726,T
1001,-0.358893,1.404453,0.704965,-0.200638,J
1002,-0.501840,0.659254,-0.421691,-0.057688,R
1003,0.204886,1.074134,1.388361,-0.982404,S
1004,0.354628,-0.133116,0.283763,-0.837063,B
...,...,...,...,...,...
1095,1.106521,0.098153,0.789793,1.192693,8
1096,-0.540543,1.782569,0.051931,0.463868,Z
1097,-0.101980,0.981720,1.106990,-1.752269,L
1098,0.632107,-0.761419,1.427930,-0.046928,E


In [27]:
chunks.get_chunk(100)

,one,two,three,four,key
1100,-0.602748,-0.182504,-0.768164,1.260686,D
1101,-0.039105,-0.069510,-1.358052,-0.292859,N
1102,-0.345101,-0.179477,-0.904658,-0.071287,D
1103,-0.759495,0.225916,2.235872,-0.004490,E
1104,-1.686771,0.742276,0.485876,1.798507,V
...,...,...,...,...,...
1195,-0.389368,-0.932914,-0.998025,0.563617,O
1196,-0.558272,-0.161775,-1.538067,0.613443,3
1197,-0.158497,1.349013,-0.475158,-1.033059,U
1198,0.407671,0.254557,-1.560825,1.125628,I


It can also be used as an iterator:

In [28]:
# Read in chunks
chunks = pd.read_csv("../examples/ex6.csv", chunksize=1000)
chunks

In [29]:
key_counts = pd.Series([])

for chunk in chunks:
    key_counts = key_counts.add(chunk["key"].value_counts(), fill_value=0)

key_counts = key_counts.sort_values
key_counts(ascending=False)

key
E    368
X    364
L    346
O    343
Q    340
    ... 
5    157
2    152
0    151
9    150
1    146
Length: 36, dtype: object

### Writing Data to Text Format

In [30]:
# Load example data
data = pd.read_csv("../examples/ex5.csv")

In [31]:
# Export to csv
data.to_csv(sys.stdout)

,something,a,b,c,d,message
0,one,1,2,3.0,4,
1,two,5,6,,8,world
2,three,9,10,11.0,12,foo


In [32]:
# Change delimiter
data.to_csv(sys.stdout, sep="\t")

	something	a	b	c	d	message
0	one	1	2	3.0	4	
1	two	5	6		8	world
2	three	9	10	11.0	12	foo


In [33]:
# Change encoding of missing values
data.to_csv(sys.stdout, na_rep="NaN")

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [34]:
# Exclude rows and column names
data.to_csv(sys.stdout, index=False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


In [35]:
# Select columns to export
data.to_csv(sys.stdout, columns=["something", "message"])

,something,message
0,one,
1,two,world
2,three,foo


### Working with Other Delimited Formats

Example: Open and import csv file with all values wrapped in quotes:

In [36]:
# Example data
with open("../examples/ex7.csv") as file:
    print(file.read())

"a","b","c"
"1","2","3"
"1","2","3"



#### Import manually

In [37]:
# Import file with base Python reader
# which removes the quotes
with open("../examples/ex7.csv") as file:
    lines = list(csv.reader(file))

lines

[['a', 'b', 'c'], ['1', '2', '3'], ['1', '2', '3']]

In [38]:
# Extract header and rest of data
headers, values = lines[0], lines[1:]

In [39]:
# Create tuple from headers
for i in zip(headers):
    print(i)

('a',)
('b',)
('c',)


In [40]:
# Create tuples from values, after transposing appropriately
for i in zip(*values):
    print(i)

('1', '1')
('2', '2')
('3', '3')


In [41]:
# Create tuple for header:value relationship
for i in zip(headers, zip(*values)):
    print(i)

pairs = zip(headers, zip(*values))

('a', ('1', '1'))
('b', ('2', '2'))
('c', ('3', '3'))


In [42]:
# Create dictionary containing data column by column using a comprehension
dict = {header: value for (header, value) in pairs}

# Create dataframe
pd.DataFrame(dict)

,a,b,c
0,1,2,3
1,1,2,3


#### Automatic parsing using custom arguments

Instead of manual operations, we can supply appropriate parameters to `csv.reader`

In [43]:
with open("../examples/ex7.csv") as file:
    df = pd.DataFrame(csv.reader(file, quotechar='"'))

df

,0,1,2
0,a,b,c
1,1,2,3
2,1,2,3


### JSON Data

The JSON format is very similar to a Python dictionary. JSON files can be read using the `json.loads` function, producing a dictionary.

In [44]:
# Example data
obj = """
{"name": "Wes",
 "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]},
              {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]
}
"""

In [45]:
result = json.loads(obj)
result

{'name': 'Wes',
 'cities_lived': ['Akron', 'Nashville', 'New York', 'San Francisco'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 34, 'hobbies': ['guitars', 'soccer']},
  {'name': 'Katie', 'age': 42, 'hobbies': ['diving', 'art']}]}

A DataFrame can then be created from the dictionary.

In [46]:
# Create DataFrame from siblings element of dictionary
pd.DataFrame(result["siblings"])

,name,age,hobbies
0,Scott,34,"[guitars, soccer]"
1,Katie,42,"[diving, art]"


In [47]:
# Create DataFrame from siblings element of dictionary, selected cols only
example = pd.DataFrame(result["siblings"], columns=["name", "hobbies"])
example

,name,hobbies
0,Scott,"[guitars, soccer]"
1,Katie,"[diving, art]"


The `pandas.read_json` function can perform this automatically. It assumes that all arrays are of the same length, with each representing one row in the dataframe.

In [48]:
# Show example data
with open("../examples/example.json") as file:
    print(file.read())

[{"a": 1, "b": 2, "c": 3},
 {"a": 4, "b": 5, "c": 6},
 {"a": 7, "b": 8, "c": 9}]



In [49]:
pd.read_json("../examples/example.json")

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


Pandas objects can be exported to JSON files using the appropriate methods.

In [50]:
example.to_json(sys.stdout)

{"name":{"0":"Scott","1":"Katie"},"hobbies":{"0":["guitars","soccer"],"1":["diving","art"]}}

### XML and HTML: Web Scraping

#### HTML Parsing

The `pandas.read_html` will attempt to parse all tables from a html page.

In [51]:
# Get list of tables
tables_html = pd.read_html("../examples/fdic_failed_bank_list.html")

In [52]:
# Get number of tables
len(tables_html)

1

In [53]:
# Assign table to variable
failures = tables_html[0]
failures

,Bank Name,City,ST,CERT,Acquiring Institution,Closing Date,Updated Date
0,Allied Bank,Mulberry,AR,91,Today's Bank,"September 23, 2016","November 17, 2016"
1,The Woodbury Banking Company,Woodbury,GA,11297,United Bank,"August 19, 2016","November 17, 2016"
2,First CornerStone Bank,King of Prussia,PA,35312,First-Citizens Bank & Trust Company,"May 6, 2016","September 6, 2016"
3,Trust Company Bank,Memphis,TN,9956,The Bank of Fayette County,"April 29, 2016","September 6, 2016"
4,North Milwaukee State Bank,Milwaukee,WI,20364,First-Citizens Bank & Trust Company,"March 11, 2016","June 16, 2016"
...,...,...,...,...,...,...,...
542,"Superior Bank, FSB",Hinsdale,IL,32646,"Superior Federal, FSB","July 27, 2001","August 19, 2014"
543,Malta National Bank,Malta,OH,6629,North Valley Bank,"May 3, 2001","November 18, 2002"
544,First Alliance Bank & Trust Co.,Manchester,NH,34264,Southern New Hampshire Bank & Trust,"February 2, 2001","February 18, 2003"
545,National State Bank of Metropolis,Metropolis,IL,3815,Banterra Bank of Marion,"December 14, 2000","March 17, 2005"


#### Parsing XML with lxml.objectify

XML data can be read with `pandas.read_xml`.

In [54]:
data = pd.read_xml("../datasets/mta_perf/Performance_MNR.xml")
data.head(5)

,INDICATOR_SEQ,PARENT_SEQ,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,CATEGORY,FREQUENCY,DESIRED_CHANGE,INDICATOR_UNIT,DECIMAL_PLACES,YTD_TARGET,YTD_ACTUAL,MONTHLY_TARGET,MONTHLY_ACTUAL
0,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,1,Service Indicators,M,U,%,1,95.00,96.90,95.00,96.90
1,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,2,Service Indicators,M,U,%,1,95.00,96.00,95.00,95.00
2,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,3,Service Indicators,M,U,%,1,95.00,96.30,95.00,96.90
3,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,4,Service Indicators,M,U,%,1,95.00,96.80,95.00,98.30
4,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,5,Service Indicators,M,U,%,1,95.00,96.60,95.00,95.80


## Binary Data Formats

### Pandas Pickle
Pandas objects can be stored using their `to_pickle` method.

In [55]:
frame = pd.read_csv("../examples/ex1.csv")
frame.to_pickle("../examples/frame_pickle.pkl")

### Microsoft Excel

Read Excel files using `pandas.ExcelFile`. From there, data from each sheet can be read into a DataFrame.

In [56]:
# Read xlsx file
excel = pd.ExcelFile("../examples/ex1.xlsx")

In [57]:
# Get list of sheet names
print(excel.sheet_names)

# Parse sheet into dataframe
excel.parse("Sheet1", index_col=0)

['Sheet1']


,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Sheets can also be read directly from an Excel file:

In [58]:
pd.read_excel("../examples/ex1.xlsx", sheet_name="Sheet1", index_col=0)

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


DataFrames can then be written to Excel files:

In [59]:
# One step
frame.to_excel("../examples/ex2.xlsx")

# By opening and closing a connection
writer = pd.ExcelWriter("../examples/ex2.xlsx")
frame.to_excel(writer, "Sheet1")
writer.close()

<positron-console-cell-59>:6: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.


### HDF5

HDF5 files can be accessed via the `HDFStore` class.

In [60]:
# Create dataframe to store
frame = pd.DataFrame({"a": np.random.standard_normal(100)})

# Open connection
store = pd.HDFStore("../examples/mydata.h5")

# Store data
## Store a DataFrame
store["obj1"] = frame

## Store a Series
store["obj1_col"] = frame["a"]

# Show
store

<class 'pandas.io.pytables.HDFStore'>
File path: ../examples/mydata.h5

These objects can be retrieved like dictionaries.

In [61]:
store["obj1"]

,a
0,0.929693
1,1.019486
2,0.189753
3,0.099025
4,0.082187
...,...
95,1.262332
96,-1.108131
97,-0.189854
98,-0.352938


In [62]:
store["obj1_col"]

0     0.929693
1     1.019486
2     0.189753
3     0.099025
4     0.082187
        ...   
95    1.262332
96   -1.108131
97   -0.189854
98   -0.352938
99    1.116192
Name: a, Length: 100, dtype: float64

## Interacting with Web APIs

The `requests` package can be used to grab data from online APIs.

In [63]:
# Example: Last 30 GitHub issues for pandas
response = requests.get("https://api.github.com/repos/pandas-dev/pandas/issues")
response.raise_for_status()

# Response 200 is good
response

<Response [200]>

Access data using the object's `json` method. In this case, it is returned as a dictionary.

In [64]:
data = response.json()
data

[{'url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414',
  'repository_url': 'https://api.github.com/repos/pandas-dev/pandas',
  'labels_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/labels{/name}',
  'comments_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/comments',
  'events_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/events',
  'html_url': 'https://github.com/pandas-dev/pandas/pull/61414',
  'id': 3050669700,
  'node_id': 'PR_kwDOAA0YD86VhcTU',
  'number': 61414,
  'title': 'Bug fix slow plot with datetimeindex',
  'user': {'login': 'thehalvo',
   'id': 1031728,
   'node_id': 'MDQ6VXNlcjEwMzE3Mjg=',
   'avatar_url': 'https://avatars.githubusercontent.com/u/1031728?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/thehalvo',
   'html_url': 'https://github.com/thehalvo',
   'followers_url': 'https://api.github.com/users/thehalvo/followers',
   'following_url': 'https://api.github.com/user

In [65]:
# Subset into data: Look into newest issue
data[0]

{'url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414',
 'repository_url': 'https://api.github.com/repos/pandas-dev/pandas',
 'labels_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/labels{/name}',
 'comments_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/comments',
 'events_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/61414/events',
 'html_url': 'https://github.com/pandas-dev/pandas/pull/61414',
 'id': 3050669700,
 'node_id': 'PR_kwDOAA0YD86VhcTU',
 'number': 61414,
 'title': 'Bug fix slow plot with datetimeindex',
 'user': {'login': 'thehalvo',
  'id': 1031728,
  'node_id': 'MDQ6VXNlcjEwMzE3Mjg=',
  'avatar_url': 'https://avatars.githubusercontent.com/u/1031728?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/thehalvo',
  'html_url': 'https://github.com/thehalvo',
  'followers_url': 'https://api.github.com/users/thehalvo/followers',
  'following_url': 'https://api.github.com/users/thehalvo/followin

In [66]:
# Create DataFrame from data
issues = pd.DataFrame(data, columns=["number", "title", "user", "labels", "state"])
issues

,number,title,user,labels,state
0,61414,Bug fix slow plot with datetimeindex,"{'login': 'thehalvo', 'id': 1031728, 'node_id'...",[],open
1,61413,GH61405 Expose arguments in DataFrame.query,"{'login': 'loicdiridollou', 'id': 52484416, 'n...",[],open
2,61412,DOC: Error in Getting started tutorials > How ...,"{'login': 'paintdog', 'id': 6173456, 'node_id'...","[{'id': 134699, 'node_id': 'MDU6TGFiZWwxMzQ2OT...",open
3,61410,"CI: Upgrade to ubuntu-24.04, install Python fr...","{'login': 'mroeschke', 'id': 10647082, 'node_i...","[{'id': 48070600, 'node_id': 'MDU6TGFiZWw0ODA3...",open
4,61407,BUG: to_csv() quotechar/escapechar behavior di...,"{'login': 'johnrtian', 'id': 21209995, 'node_i...","[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
...,...,...,...,...,...
25,61370,ENH: Adding hint to to_sql,"{'login': 'AliKayhanAtay', 'id': 34006392, 'no...","[{'id': 76812, 'node_id': 'MDU6TGFiZWw3NjgxMg=...",open
26,61368,BUG: Python 3.14 may not increment refcount,"{'login': 'tacaswell', 'id': 199813, 'node_id'...","[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
27,61365,BUG: Constructing series with Timedelta object...,"{'login': 'Casper-Guo', 'id': 89810860, 'node_...","[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
28,61362,QST: best way to extend/subclass pandas.DataFrame,"{'login': 'rwijtvliet', 'id': 4106013, 'node_i...","[{'id': 34444536, 'node_id': 'MDU6TGFiZWwzNDQ0...",open


## Interacting with Databases

### SQL via `sqlite3`
Python can be used to create and interact with SQL databases using the `sqlite3` module.

Data from a SQL database is returned as a list of tuples.

In [67]:
# Get data
con = sqlite3.connect("../examples/mydata.sqlite")
cursor = con.execute("SELECT * FROM test")

# Data
data = cursor.fetchall()

# Descriptions
desc = cursor.description

# Close connection
con.close()

In [68]:
# Show data
data

[('Atlanta', 'Georgia', 1.25, 6),
 ('Tallahassee', 'Florida', 2.6, 3),
 ('Sacramento', 'California', 1.7, 5)]

In [69]:
# Show descriptions
desc

(('a', None, None, None, None, None, None),
 ('b', None, None, None, None, None, None),
 ('c', None, None, None, None, None, None),
 ('d', None, None, None, None, None, None))

In [70]:
# Extract relevant info (first element of each tuple)
[column[0] for column in desc]

['a', 'b', 'c', 'd']

In [71]:
# Use data and column names to create dataframe
pd.DataFrame(data, columns=[column[0] for column in desc])

,a,b,c,d
0,Atlanta,Georgia,1.25,6
1,Tallahassee,Florida,2.60,3
2,Sacramento,California,1.70,5


### SQL via `sqlalchemy`
SQL databases can also be accessed via `pandas` and `sqlalchemy`.

In [72]:
engine = sqla.create_engine("sqlite:///../examples/mydata.sqlite")
df = pd.read_sql("SELECT * FROM test", engine)
engine.dispose()

In [73]:
# Show data
df

,a,b,c,d
0,Atlanta,Georgia,1.25,6
1,Tallahassee,Florida,2.60,3
2,Sacramento,California,1.70,5
